# PointPillars trên nuScenes — chấm điểm theo chuẩn nuScenes

Notebook chạy **PointPillars đã huấn luyện sẵn** (MMDetection3D) trên `v1.0-mini` và:

1. **Chấm điểm đúng chuẩn nuScenes** (mAP, NDS) bằng bộ chấm chính thức `DetectionEval` của devkit, trên split `mini_val` (scene-0103, scene-0916, 81 keyframe).
2. **So sánh hai cách đưa dữ liệu vào model**:
   - **1 lần quét**: chỉ dùng lần quét lidar của keyframe. Đây là cách cũ, khoảng 35 nghìn điểm.
   - **11 lần quét**: keyframe cộng 10 lần quét liền trước, **giống lúc model được huấn luyện**, khoảng 300 nghìn điểm.
3. **Hiển thị hai hình 3D cạnh nhau** để xem dự đoán và nhãn gốc, dùng đúng các box mà bộ chấm điểm đã lọc.

| Bên trái | Bên phải |
|---|---|
| **NHÃN GỐC** (ground truth) | **DỰ ĐOÁN** của PointPillars |
| Xanh lá = model tìm ra (TP) | Xanh lá = đúng (TP) |
| Cam = model bỏ sót (FN) | Đỏ = báo nhầm (FP) |

> ⚠️ **Bắt buộc dùng runtime có GPU.** Trên Colab: menu **Runtime → Change runtime type → T4 GPU → Save**.
>
> Colab hiện chạy Python 3.13, quá mới với MMDetection3D. Vì vậy notebook cài MMDetection3D vào **một môi trường Python 3.10 riêng** và gọi nó để suy luận. Phần đọc dữ liệu, chấm điểm và vẽ vẫn chạy trong kernel Colab bình thường.

**Dữ liệu lấy từ Google Drive:** chỉ cần file `v1.0-mini.tgz` nằm trong Drive của bạn. Cell 3 tự mount Drive và giải nén phần cần thiết vào `/content/nuscenes`.

Chạy lần lượt từ trên xuống (**Runtime → Run all** cũng được, nhưng Cell 3 sẽ hỏi quyền truy cập Drive).

## Cell 1 — Kiểm tra GPU

Nếu báo lỗi không có GPU: menu **Runtime → Change runtime type → T4 GPU → Save**, rồi chạy lại cell này.

In [1]:
import importlib.util, subprocess, sys

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(
        'Runtime nay KHONG co GPU. Vao menu Runtime -> Change runtime type -> T4 GPU -> Save, '
        'roi chay lai tu cell nay.')

print('GPU    :', r.stdout.strip())
print('Python :', sys.version.split()[0])
assert importlib.util.find_spec('google.colab') is not None, \
    'Notebook nay danh cho Google Colab (colab.research.google.com).'
print('Kernel : Colab OK')

GPU    : Tesla T4, 15360 MiB, 580.82.07
Python : 3.13.15
Kernel : Colab OK


## Cell 2 — Cài đặt

Colab giờ chạy **Python 3.13**, còn bộ MMDetection3D (mmcv 2.x, mmdet 3.x, mmdet3d 1.4) chỉ hỗ trợ tới Python 3.11. Cài thẳng vào kernel Colab sẽ hỏng ở nhiều chỗ: không có torchvision phù hợp, `openmim` kéo `setuptools` về bản 60 không chạy được trên Python 3.12+, và `mmcv` phải build từ source.

Cách làm ở đây:
- **Kernel Colab** chỉ cài `nuscenes-devkit` và `plotly` để đọc dữ liệu và vẽ. `nuscenes-devkit` được cài **không kèm phụ thuộc**, vì nó khai báo cần numpy dưới bản 2 và sẽ hạ numpy của Colab, làm hỏng opencv. Thực tế phần devkit mà notebook dùng chạy tốt với numpy 2.
- **MMDetection3D** được cài vào môi trường riêng `/content/mmenv` chạy **Python 3.10**, tạo bằng `uv`. Ở đó dùng tổ hợp đã được kiểm chứng và **có sẵn wheel**: PyTorch 2.1.2 + CUDA 11.8, mmcv 2.1.0, mmdet 3.3.0, mmdet3d 1.4.0. Không phải build gì, không phải vá giới hạn phiên bản. `setuptools` được ghim dưới bản 70 vì mmengine và mim còn dùng `pkg_resources`.

Mất khoảng **5–8 phút**. Mỗi bước được kiểm tra ngay, lỗi ở bước nào sẽ dừng và in log của bước đó. Log đầy đủ ở `/content/install.log`.

Nếu một lần cài trước đó đã hạ numpy của kernel, cell này sẽ nâng lại lên numpy 2. Khi đó cell sẽ báo **cần restart**: vào menu **Runtime → Restart session**, rồi chạy lại từ Cell 1. Lần chạy lại cell này chỉ mất vài giây vì phần đã cài được bỏ qua.

In [2]:
import os, subprocess, sys, time

ENV  = '/content/mmenv'           # moi truong Python 3.10 rieng cho MMDetection3D
PY   = f'{ENV}/bin/python'
LOG  = '/content/install.log'
DONE = f'{ENV}/.install_ok'


def run(desc, cmd):
    """Chay mot lenh cai dat; dung ngay va in log neu lenh that bai."""
    t0 = time.time()
    print(f'>> {desc} ...', end=' ', flush=True)
    with open(LOG, 'a') as f:
        f.write(f'\n\n$ {cmd}\n')
        f.flush()
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print('LOI\n')
        print(open(LOG).read()[-4000:])
        raise RuntimeError(f'Buoc "{desc}" that bai. Log day du: {LOG}')
    print(f'xong ({time.time() - t0:.0f}s)')


# 1) Kernel Colab: devkit de doc du lieu, plotly de ve, uv de tao moi truong rieng.
#    --no-deps: devkit khai bao numpy<2 nhung thuc te chay tot voi numpy 2. Cai kem phu thuoc
#    se ha numpy cua Colab xuong 1.26 va lam hong opencv, jax... (opencv moi can numpy>=2).
#    Colab da co san cv2, matplotlib, sklearn, PIL, tqdm, cachetools; chi thieu pyquaternion.
run('Cai nuscenes-devkit (khong kem phu thuoc) + pyquaternion',
    f'{sys.executable} -m pip install -q --no-deps nuscenes-devkit pyquaternion')
# Giu numpy/shapely o ban ma Colab can (sua lai neu lan cai truoc da ha phien ban).
run('Cai plotly, uv; giu numpy 2 va shapely 2.1 cho kernel',
    f'{sys.executable} -m pip install -q plotly uv "numpy>=2.1,<2.6" "shapely>=2.1"')

# 2) Moi truong Python 3.10 rieng cho MMDetection3D.
if os.path.exists(DONE):
    print('Moi truong MMDetection3D da cai truoc do, bo qua.')
else:
    UV = f'{sys.executable} -m uv pip install --python {PY}'
    MMCV_WHEELS = 'https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html'

    run('Tao moi truong Python 3.10',
        f'{sys.executable} -m uv venv {ENV} --python 3.10 --seed --clear')
    run('Cai PyTorch 2.1.2 + CUDA 11.8',
        f'{UV} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118')
    # --only-binary mmcv: bat buoc dung wheel dung san, khong bao gio build tu source.
    # setuptools<70: mmengine va mim can pkg_resources, ban setuptools moi da bo module nay.
    PINS = '"numpy<2" "setuptools<70"'
    run('Cai mmengine + mmcv 2.1.0',
        f'{UV} {PINS} "mmengine>=0.10.3,<0.11" mmcv==2.1.0 '
        f'--find-links {MMCV_WHEELS} --only-binary mmcv')
    run('Cai mmdet 3.3.0 + mmdet3d 1.4.0 + openmim',
        f'{UV} {PINS} mmdet==3.3.0 mmdet3d==1.4.0 openmim '
        f'--find-links {MMCV_WHEELS} --only-binary mmcv')
    open(DONE, 'w').close()

# 3) Neu pip vua doi phien ban numpy ma kernel da nap ban khac, phai restart kernel.
if 'numpy' in sys.modules:
    on_disk = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                             capture_output=True, text=True).stdout.strip()
    loaded = sys.modules['numpy'].__version__
    if on_disk != loaded:
        raise RuntimeError(f'CAN RESTART KERNEL: numpy da doi tu {loaded} sang {on_disk}. '
                         'Vao menu Runtime -> Restart session, roi chay lai tu Cell 1.')

print('\nCai dat xong. Chay tiep cell kiem tra ben duoi.')

>> Cai nuscenes-devkit (khong kem phu thuoc) + pyquaternion ... xong (3s)
>> Cai plotly, uv; giu numpy 2 va shapely 2.1 cho kernel ... xong (7s)
>> Tao moi truong Python 3.10 ... xong (3s)
>> Cai PyTorch 2.1.2 + CUDA 11.8 ... xong (46s)
>> Cai mmengine + mmcv 2.1.0 ... xong (7s)
>> Cai mmdet 3.3.0 + mmdet3d 1.4.0 + openmim ... xong (17s)

Cai dat xong. Chay tiep cell kiem tra ben duoi.


In [3]:
import subprocess, sys

ENV = '/content/mmenv'
PY  = f'{ENV}/bin/python'

# Kiem tra moi truong MMDetection3D (Python 3.10).
CHECK = """
import sys, torch, numpy, mmengine, mmcv, mmdet, mmdet3d
from mmcv.ops import Voxelization          # op CUDA ma PointPillars can
print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('numpy   ', numpy.__version__)
print('mmengine', mmengine.__version__)
print('mmcv    ', mmcv.__version__)
print('mmdet   ', mmdet.__version__)
print('mmdet3d ', mmdet3d.__version__)
assert torch.cuda.is_available(), 'Khong thay GPU: chon server Colab co GPU T4 (xem Cell 1)'
"""
r = subprocess.run([PY, '-c', CHECK], capture_output=True, text=True)
print('--- Moi truong MMDetection3D ---')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise RuntimeError('Moi truong MMDetection3D chua dung. Xem loi o tren, '
                       'hoac xoa /content/mmenv roi chay lai Cell 2.')

# Kiem tra kernel Colab.
import numpy, plotly, nuscenes
print('--- Kernel Colab ---')
print('python  ', sys.version.split()[0])
print('numpy   ', numpy.__version__)
print('plotly  ', plotly.__version__)
print('nuscenes-devkit OK')

--- Moi truong MMDetection3D ---
python   3.10.21
torch    2.1.2+cu118 | CUDA: True
numpy    1.26.4
mmengine 0.10.7
mmcv     2.1.0
mmdet    3.3.0
mmdet3d  1.4.0

--- Kernel Colab ---
python   3.13.15
numpy    2.1.3
plotly   5.24.1
nuscenes-devkit OK


## Cell 3 — Dữ liệu nuScenes từ Google Drive

Cell này:
1. **Mount Google Drive**. Lần đầu Colab sẽ hỏi quyền truy cập, chọn tài khoản chứa file và bấm **Allow**.
2. Tìm `v1.0-mini.tgz` theo đường dẫn `TGZ_PATH`. Nếu không có ở đó, cell tự tìm trong toàn bộ **My Drive**.
3. **Giải nén chỉ phần notebook cần** vào `/content/nuscenes` (ổ đĩa của runtime, đọc nhanh hơn Drive nhiều):
   - `v1.0-mini/*.json`: bảng annotation và metadata
   - `samples/LIDAR_TOP`: 404 lần quét lidar của các keyframe
   - `sweeps/LIDAR_TOP`: các lần quét xen giữa keyframe, dùng để gộp 10 sweep
   - `maps/*.png`: devkit cần các file này khi khởi tạo

   Ảnh camera và radar được bỏ qua. Vẫn phải đọc hết file 4 GB nên mất khoảng **3–6 phút**. Lần chạy lại trên cùng runtime sẽ bỏ qua bước này.

Nếu file của bạn nằm ở chỗ khác, sửa `TGZ_PATH` (ví dụ `/content/drive/MyDrive/datasets/v1.0-mini.tgz`).

Dữ liệu trong `/content` mất khi runtime bị thu hồi. File trên Drive vẫn còn, chỉ cần chạy lại cell này.

In [ ]:
import glob, os, subprocess, time, os.path as osp
from google.colab import drive
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes

TGZ_PATH = '/content/drive/MyDrive/AI_Action/Data/v1.0-mini.tgz'  #@param {type:"string"}
DATAROOT = '/content/nuscenes'     # giai nen vao o dia cua runtime, doc nhanh hon Drive
N_LIDAR_EXPECTED = 404             # so file lidar keyframe cua v1.0-mini
N_SWEEPS = 10                      # so lan quet gop them truoc keyframe, giong luc huan luyen
DONE = osp.join(DATAROOT, '.extract_ok')

# Chi lay phan notebook can: bo anh camera va radar (phan lon dung luong cua file .tgz).
MEMBERS = ['v1.0-mini/*', 'maps/*', 'samples/LIDAR_TOP/*', 'sweeps/LIDAR_TOP/*']

if osp.exists(DONE):
    print('Du lieu da giai nen truoc do, bo qua.')
else:
    if not osp.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    if not osp.isfile(TGZ_PATH):
        print(f'Khong thay {TGZ_PATH}, dang tim v1.0-mini.tgz trong My Drive ...')
        found = glob.glob('/content/drive/MyDrive/**/v1.0-mini.tgz', recursive=True)
        if not found:
            raise FileNotFoundError(
                'Khong tim thay v1.0-mini.tgz trong My Drive. Kiem tra file da upload xong chua, '
                'hoac sua TGZ_PATH cho dung duong dan roi chay lai cell nay.')
        TGZ_PATH = found[0]
    print(f'Giai nen {TGZ_PATH} ({osp.getsize(TGZ_PATH) / 1e9:.1f} GB) vao {DATAROOT} ...')
    t0 = time.time()
    os.makedirs(DATAROOT, exist_ok=True)
    r = subprocess.run(['tar', '-xzf', TGZ_PATH, '-C', DATAROOT, '--wildcards', *MEMBERS],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError('Giai nen that bai. File .tgz tren Drive co the bi hong hoac chua upload xong.')
    print(f'Giai nen xong ({time.time() - t0:.0f}s)')

n_lidar = len(glob.glob(osp.join(DATAROOT, 'samples', 'LIDAR_TOP', '*.pcd.bin')))
n_maps = len(glob.glob(osp.join(DATAROOT, 'maps', '*.png')))
print(f'File lidar keyframe: {n_lidar}/{N_LIDAR_EXPECTED} | file ban do: {n_maps}/4')
assert n_lidar == N_LIDAR_EXPECTED and n_maps == 4, \
    'Thieu file sau khi giai nen: xoa /content/nuscenes (!rm -rf /content/nuscenes) roi chay lai cell nay.'

nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=False)
print('Nap xong:', len(nusc.scene), 'scene,', len(nusc.sample), 'keyframe')

# Moi keyframe cua mini_val can N_SWEEPS lan quet lien truoc no.
needed = set()
for s in nusc.sample:
    if nusc.get('scene', s['scene_token'])['name'] not in create_splits_scenes()['mini_val']:
        continue
    sd = nusc.get('sample_data', s['data']['LIDAR_TOP'])
    for _ in range(N_SWEEPS):
        if not sd['prev']:
            break
        sd = nusc.get('sample_data', sd['prev'])
        needed.add(sd['filename'])
missing = sorted(f for f in needed if not osp.exists(osp.join(DATAROOT, f)))
print(f'Lan quet can cho mini_val: {len(needed) - len(missing)}/{len(needed)} co tren runtime')
for f in missing[:10]:
    print('  thieu:', f)
assert not missing, (f'Thieu {len(missing)} file sweep: xoa /content/nuscenes '
                     '(!rm -rf /content/nuscenes) roi chay lai cell nay.')
open(DONE, 'w').close()

Mounted at /content/drive
Khong thay /content/drive/MyDrive/AI_Action/Data/v1.0-mini.tgz, dang tim v1.0-mini.tgz trong My Drive ...


## Cell 4 — Tải trọng số PointPillars đã huấn luyện

Dùng `mim download` để lấy đúng file cấu hình và trọng số khớp với phiên bản mmdet3d đang cài. Mô hình: **PointPillars + FPN**, huấn luyện trên tập train đầy đủ của nuScenes (700 scene), 10 lớp.

In [ ]:
import glob, subprocess

ENV = '/content/mmenv'
CKPT_DIR = '/content/checkpoints'
MODEL_NAME = 'pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d'

if not glob.glob(f'{CKPT_DIR}/*.pth'):
    subprocess.run([f'{ENV}/bin/mim', 'download', 'mmdet3d',
                    '--config', MODEL_NAME, '--dest', CKPT_DIR], check=True)

cfg_file  = glob.glob(f'{CKPT_DIR}/*.py')[0]
ckpt_file = glob.glob(f'{CKPT_DIR}/*.pth')[0]
print('config    :', cfg_file)
print('checkpoint:', ckpt_file)

## Cell 5 — Chuẩn bị bộ suy luận

Model chạy trong môi trường Python 3.10, nên cell này ghi ra một script nhỏ `/content/pp_infer.py`. Notebook gọi script đó với danh sách file point cloud và nhận lại box qua file `.npz`. Mỗi lần gọi, model chỉ nạp một lần rồi suy luận lần lượt cho tất cả các file.

Pipeline gốc của mmdet3d gộp sweep bằng `LoadPointsFromMultiSweeps`, nhưng bước này cần file annotation `.pkl` riêng của mmdet3d. Vì vậy **notebook tự gộp sweep ở Cell 6, theo đúng cách của bước đó**, rồi bỏ bước đó khỏi pipeline. Mỗi điểm đưa vào model có 4 cột `[0, 1, 2, 4]` = `(x, y, z, Δt)`. Model nuScenes này **không dùng cường độ phản xạ** (cột 3).

Script trả về cho mỗi box: tâm, kích thước, góc xoay, **vận tốc** (model dự đoán được khi có nhiều lần quét) và 8 góc hộp.

Cell này nạp thử model một lần để kiểm tra, mất khoảng 10–20 giây.

In [ ]:
import subprocess, numpy as np

ENV = '/content/mmenv'
PY = f'{ENV}/bin/python'
WORKER = '/content/pp_infer.py'

WORKER_CODE = r"""
# Chay trong moi truong Python 3.10.  Cach goi:
#   python pp_infer.py CONFIG CHECKPOINT OUT.npz [frame1.bin frame2.bin ...]
# Moi file .bin la float32 (N, 5) = (x, y, z, intensity, dt), da gop sweep san.
import sys
import numpy as np
from mmdet3d.apis import init_model, inference_detector

cfg_file, ckpt_file, out_file = sys.argv[1:4]
bins = sys.argv[4:]

model = init_model(cfg_file, ckpt_file, device='cuda:0')

# Bo buoc gop sweep khoi pipeline (notebook da gop san), nhung giu cach no chon cot:
# [0, 1, 2, 4] = (x, y, z, dt). Model nuScenes nay KHONG dung intensity (cot 3).
full = model.cfg.test_dataloader.dataset.pipeline
sweeps = [t for t in full if t['type'] == 'LoadPointsFromMultiSweeps']
use_dim = list(sweeps[0].get('use_dim', [0, 1, 2, 4])) if sweeps else [0, 1, 2, 4]
sweeps_num = int(sweeps[0].get('sweeps_num', 10)) if sweeps else 0
n_in = int(model.cfg.model.pts_voxel_encoder.get('in_channels', 4))
assert len(use_dim) == n_in, f'use_dim {use_dim} khong khop in_channels={n_in} cua model'

pipeline = [t for t in full if t['type'] != 'LoadPointsFromMultiSweeps']
for t in pipeline:
    if t['type'] == 'LoadPointsFromFile':
        t['load_dim'], t['use_dim'] = 5, use_dim
model.cfg.test_dataloader.dataset.pipeline = pipeline

out = {'classes': np.array(model.dataset_meta['classes']),
       'pipeline': np.array([t['type'] for t in pipeline]),
       'use_dim': np.array(use_dim),
       'sweeps_num': np.array(sweeps_num)}
for i, path in enumerate(bins):
    result, _ = inference_detector(model, path)
    p = result.pred_instances_3d
    b = p.bboxes_3d                                                  # he toa do lidar
    vel = b.tensor[:, 7:9] if b.tensor.shape[1] >= 9 else b.tensor.new_zeros((len(b), 2))
    out[f'centers_{i}'] = b.gravity_center.cpu().numpy()             # (N, 3) tam hinh hoc
    out[f'dims_{i}'] = b.dims.cpu().numpy()                          # (N, 3) = (dai, rong, cao)
    out[f'yaws_{i}'] = b.yaw.cpu().numpy()                           # (N,) goc quanh truc z
    out[f'vels_{i}'] = vel.cpu().numpy()                             # (N, 2) van toc vx, vy
    out[f'corners_{i}'] = b.corners.cpu().numpy()                    # (N, 8, 3)
    out[f'labels_{i}'] = p.labels_3d.cpu().numpy()
    out[f'scores_{i}'] = p.scores_3d.cpu().numpy()
np.savez(out_file, **out)
"""
with open(WORKER, 'w') as f:
    f.write(WORKER_CODE)


def call_worker(out_file, bins=()):
    """Goi script suy luan trong moi truong Python 3.10, tra ve noi dung file .npz."""
    r = subprocess.run([PY, WORKER, cfg_file, ckpt_file, out_file, *bins],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-4000:])
        raise RuntimeError('Suy luan that bai, xem log o tren.')
    return np.load(out_file)


meta = call_worker('/content/_meta.npz')
CLASSES = [str(c) for c in meta['classes']]
print('10 lop model du doan:', CLASSES)
print('\nPipeline suy luan:', [str(t) for t in meta['pipeline']])
print('Cot dau vao moi diem:', meta['use_dim'].tolist(), '= (x, y, z, dt)')
print('So sweep luc huan luyen:', int(meta['sweeps_num']))
assert int(meta['sweeps_num']) == N_SWEEPS, \
    f"Config dung {int(meta['sweeps_num'])} sweep: sua N_SWEEPS o Cell 3 cho khop roi chay lai tu Cell 3"

## Cell 6 — Các hàm xử lý

- `load_points()` — **gộp lần quét của keyframe với N lần quét liền trước**, làm giống hệt `LoadPointsFromMultiSweeps` của mmdet3d:
  - Keyframe giữ nguyên, `Δt = 0`.
  - Với mỗi lần quét cũ hơn: bỏ các điểm trong ô vuông 1 m quanh cảm biến (điểm rơi trên nóc xe), chiếu về hệ toạ độ lidar của keyframe (vì xe đã di chuyển giữa hai lần quét), rồi đặt `Δt` = số giây mà lần quét đó cũ hơn keyframe.
  - Với `n_sweeps = 0`, hàm chỉ đọc lần quét của keyframe, như bản cũ của notebook.
- `run_pointpillars()` — suy luận nhiều keyframe trong một lần gọi, có lưu đệm.
- `check_box_convention()` — kiểm tra 8 góc hộp sau khi đổi quy ước **trùng khớp** với 8 góc do mmdet3d tính. Nếu quy ước kích thước hoặc góc xoay bị sai, mọi chỉ số chấm điểm sẽ vô nghĩa, nên notebook dừng ngay.
- `pred_to_global()` — đổi dự đoán từ hệ lidar sang **hệ toạ độ toàn cục** theo định dạng file kết quả của nuScenes, giống `NuScenesMetric` của mmdet3d. Hàm này cũng suy ra thuộc tính (đang đỗ, đang chạy...) từ vận tốc.
- `global_to_lidar_corners()` — chiều ngược lại, dùng để vẽ box cùng point cloud.
- `match_boxes()` — so khớp theo chuẩn nuScenes: cùng lớp, tâm cách nhau dưới 2 m trên mặt phẳng ngang, ưu tiên dự đoán có điểm tin cậy cao.

In [ ]:
import os
import numpy as np
from pyquaternion import Quaternion
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import transform_matrix

FRAME_DIR = '/content/_frames'
os.makedirs(FRAME_DIR, exist_ok=True)
PRED_KEYS = ('centers', 'dims', 'yaws', 'vels', 'corners', 'labels', 'scores')
_PRED_CACHE = {}   # (sample token, so sweep) -> du doan tho cua model

# Thuoc tinh mac dinh theo lop, giong NuScenesMetric cua mmdet3d (model khong du doan thuoc tinh).
DEFAULT_ATTR = {'car': 'vehicle.parked', 'pedestrian': 'pedestrian.moving', 'trailer': 'vehicle.parked',
                'truck': 'vehicle.parked', 'bus': 'vehicle.moving', 'motorcycle': 'cycle.without_rider',
                'construction_vehicle': 'vehicle.parked', 'bicycle': 'cycle.without_rider',
                'barrier': '', 'traffic_cone': ''}


def get_sample(scene_name, index):
    scene = next(s for s in nusc.scene if s['name'] == scene_name)
    assert index < scene['nbr_samples'], \
        f"{scene_name} chi co {scene['nbr_samples']} keyframe"
    token = scene['first_sample_token']
    for _ in range(index):
        token = nusc.get('sample', token)['next']
    return nusc.get('sample', token), scene


def _lidar_poses(sample_data):
    """(calibrated_sensor, ego_pose) cua mot lan quet lidar."""
    return (nusc.get('calibrated_sensor', sample_data['calibrated_sensor_token']),
            nusc.get('ego_pose', sample_data['ego_pose_token']))


def _sensor_to_global(sample_data):
    cs, pose = _lidar_poses(sample_data)
    return (transform_matrix(pose['translation'], Quaternion(pose['rotation']))
            @ transform_matrix(cs['translation'], Quaternion(cs['rotation'])))


def load_points(sample, n_sweeps):
    """Point cloud (N, 5) = (x, y, z, intensity, dt) trong he toa do lidar cua keyframe.

    Lam giong LoadPointsFromMultiSweeps cua mmdet3d: keyframe giu nguyen voi dt = 0. Voi moi sweep
    truoc do: bo diem trong o vuong 1 m quanh cam bien, chieu ve he lidar cua keyframe
    (xe da di chuyen giua hai lan quet), dt = keyframe som hon sweep bao nhieu giay.
    """
    ref = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    pts = np.fromfile(nusc.get_sample_data_path(ref['token']), dtype=np.float32).reshape(-1, 5).copy()
    pts[:, 4] = 0.0
    parts = [pts]
    global_to_ref = np.linalg.inv(_sensor_to_global(ref))
    sd = ref
    for _ in range(n_sweeps):
        if not sd['prev']:              # dau scene: co it sweep hon, model van chay duoc
            break
        sd = nusc.get('sample_data', sd['prev'])
        p = np.fromfile(nusc.get_sample_data_path(sd['token']), dtype=np.float32).reshape(-1, 5).copy()
        p = p[~((np.abs(p[:, 0]) < 1.0) & (np.abs(p[:, 1]) < 1.0))]
        tf = global_to_ref @ _sensor_to_global(sd)
        p[:, :3] = p[:, :3] @ tf[:3, :3].T + tf[:3, 3]
        p[:, 4] = (ref['timestamp'] - sd['timestamp']) * 1e-6
        parts.append(p)
    return np.concatenate(parts)


def run_pointpillars(samples, n_sweeps):
    """Du doan tho (chua loc) cho nhieu keyframe. Keyframe chua co trong bo dem duoc suy luan
    trong mot lan goi, nen model chi phai nap mot lan."""
    todo = [s for s in samples if (s['token'], n_sweeps) not in _PRED_CACHE]
    if todo:
        bins = []
        for s in todo:
            path = f"{FRAME_DIR}/{s['token']}_{n_sweeps}.bin"
            load_points(s, n_sweeps).tofile(path)
            bins.append(path)
        d = call_worker(f'{FRAME_DIR}/_batch.npz', bins)
        for i, s in enumerate(todo):
            _PRED_CACHE[(s['token'], n_sweeps)] = {k: d[f'{k}_{i}'] for k in PRED_KEYS}
        for path in bins:
            os.remove(path)
    return [_PRED_CACHE[(s['token'], n_sweeps)] for s in samples]


def _lidar_box(center, dims, yaw, vel=(0.0, 0.0)):
    """Box mmdet3d (dims = l, w, h; yaw quanh truc z) -> Box cua devkit (wlh), van trong he lidar."""
    return Box(center, dims[[1, 0, 2]], Quaternion(axis=[0, 0, 1], radians=float(yaw)),
               velocity=(float(vel[0]), float(vel[1]), 0.0))


def check_box_convention(pred, tol=0.01):
    """8 goc cua Box sau khi doi phai trung 8 goc do mmdet3d tinh. Neu khong, quy uoc kich thuoc
    hoac goc xoay bi sai va moi chi so cham diem se vo nghia."""
    for c, dim, yaw, k in zip(pred['centers'], pred['dims'], pred['yaws'], pred['corners']):
        ours = _lidar_box(c, dim, yaw).corners().T                       # (8, 3)
        gap = np.linalg.norm(ours[:, None, :] - k[None, :, :], axis=2).min(axis=1).max()
        assert gap < tol, f'Quy uoc box lech {gap:.3f} m so voi mmdet3d'
    return len(pred['centers'])


def _attribute(name, vel):
    """Giong NuScenesMetric cua mmdet3d: suy thuoc tinh tu toc do du doan."""
    if np.hypot(vel[0], vel[1]) > 0.2:
        if name in ('car', 'construction_vehicle', 'bus', 'truck', 'trailer'):
            return 'vehicle.moving'
        if name in ('bicycle', 'motorcycle'):
            return 'cycle.with_rider'
    else:
        if name == 'pedestrian':
            return 'pedestrian.standing'
        if name == 'bus':
            return 'vehicle.stopped'
    return DEFAULT_ATTR[name]


def pred_to_global(pred, sample):
    """Du doan (he lidar) -> danh sach box theo dinh dang file ket qua cua nuScenes (he toan cuc)."""
    cs, pose = _lidar_poses(nusc.get('sample_data', sample['data']['LIDAR_TOP']))
    out = []
    for c, dim, yaw, v, lab, sc in zip(pred['centers'], pred['dims'], pred['yaws'],
                                       pred['vels'], pred['labels'], pred['scores']):
        box = _lidar_box(c, dim, yaw, v)
        box.rotate(Quaternion(cs['rotation']))
        box.translate(np.array(cs['translation']))
        box.rotate(Quaternion(pose['rotation']))
        box.translate(np.array(pose['translation']))
        name = CLASSES[int(lab)]
        out.append({'sample_token': sample['token'],
                    'translation': box.center.tolist(),
                    'size': box.wlh.tolist(),
                    'rotation': box.orientation.elements.tolist(),
                    'velocity': box.velocity[:2].tolist(),
                    'detection_name': name,
                    'detection_score': float(sc),
                    'attribute_name': _attribute(name, v)})
    return out


def global_to_lidar_corners(eval_boxes, sample):
    """Box cua bo cham diem (he toan cuc) -> 8 goc trong he lidar cua keyframe, de ve cung point cloud."""
    cs, pose = _lidar_poses(nusc.get('sample_data', sample['data']['LIDAR_TOP']))
    corners = []
    for b in eval_boxes:
        box = Box(b.translation, b.size, Quaternion(b.rotation))
        box.translate(-np.array(pose['translation']))
        box.rotate(Quaternion(pose['rotation']).inverse)
        box.translate(-np.array(cs['translation']))
        box.rotate(Quaternion(cs['rotation']).inverse)
        corners.append(box.corners().T)
    return np.array(corners).reshape(-1, 8, 3)


def match_boxes(pred, gt, dist_th=2.0):
    """Khop tham lam theo diem tin cay giam dan, giong nuScenes: cung lop, tam cach nhau <= dist_th
    duoi dist_th tren mat phang ngang. pred, gt la danh sach DetectionBox."""
    pred_tp = np.zeros(len(pred), dtype=bool)
    gt_hit = np.zeros(len(gt), dtype=bool)
    pairs = []
    for pi in np.argsort([-b.detection_score for b in pred], kind='stable'):
        p = pred[pi]
        best, best_d = -1, dist_th
        for gi, g in enumerate(gt):
            if gt_hit[gi] or g.detection_name != p.detection_name:
                continue
            d = np.linalg.norm(np.array(p.translation[:2]) - np.array(g.translation[:2]))
            if d < best_d:
                best, best_d = gi, d
        if best >= 0:
            pred_tp[pi], gt_hit[best] = True, True
            pairs.append((pi, best, best_d))
    return pred_tp, gt_hit, pairs


print('Da dinh nghia xong cac ham.')

## Cell 7 — Chấm điểm chuẩn nuScenes trên `mini_val`

Cell này suy luận trên **toàn bộ 81 keyframe** của `mini_val` hai lần: chỉ với keyframe (1 lần quét), và với keyframe cộng 10 lần quét trước đó (11 lần quét). Mỗi lần ghi kết quả ra file `results_nusc.json` rồi chấm bằng **`DetectionEval` của devkit**, là đúng bộ chấm mà nuScenes dùng cho bảng xếp hạng. Mất khoảng 3–5 phút.

Bộ chấm tự lọc nhãn gốc lẫn dự đoán theo chuẩn:
- **Phạm vi theo lớp**: car, truck, bus, trailer, construction_vehicle 50 m; pedestrian, motorcycle, bicycle 40 m; traffic_cone, barrier 30 m.
- Bỏ vật **không có điểm lidar hoặc radar nào**.
- Bỏ xe đạp và xe máy đang đỗ **trong bãi xe đạp**.

**Các chỉ số:**

| Chỉ số | Ý nghĩa |
|---|---|
| **mAP** | Độ chính xác trung bình trên 10 lớp. Khớp khi tâm cách nhau dưới 0.5 / 1 / 2 / 4 m, lấy trung bình bốn mức. Tính trên mọi ngưỡng điểm tin cậy nên **không phụ thuộc** ngưỡng bạn chọn |
| **NDS** | Điểm tổng hợp của nuScenes: một nửa là mAP, nửa còn lại từ 5 loại sai số bên dưới |
| mATE, mASE, mAOE | Sai số vị trí (m), kích thước (1 − IoU), góc xoay (rad) của các box khớp đúng |
| mAVE | Sai số vận tốc (m/s). Chỉ có nghĩa khi có nhiều lần quét |
| mAAE | Tỉ lệ sai thuộc tính (đang đỗ / đang chạy, có người lái / không...) |

**Lưu ý với `mini_val`:** hai scene này **không có** trailer, construction_vehicle hay barrier. Theo quy tắc của nuScenes, lớp vắng mặt bị tính AP = 0 và sai số tối đa. Vì thế **mAP cao nhất có thể đạt được trên `mini_val` chỉ là 0.70**. Bảng có thêm dòng mAP tính riêng trên các lớp có mặt để dễ đọc.

In [ ]:
import json, os, time
from nuscenes.eval.detection.config import config_factory
from nuscenes.eval.detection.evaluate import DetectionEval
from nuscenes.utils.splits import create_splits_scenes

EVAL_SET = 'mini_val'                               # split chuan cua v1.0-mini: scene-0103, scene-0916
EVAL_CFG = config_factory('detection_cvpr_2019')    # cau hinh cham diem chinh thuc cua nuScenes
VAL_SCENES = create_splits_scenes()[EVAL_SET]
VAL_SAMPLES = [s for s in nusc.sample
               if nusc.get('scene', s['scene_token'])['name'] in VAL_SCENES]


def evaluate(n_sweeps):
    """Suy luan tren moi keyframe cua EVAL_SET roi cham bang DetectionEval cua devkit."""
    t0 = time.time()
    preds = run_pointpillars(VAL_SAMPLES, n_sweeps)
    n_checked = sum(check_box_convention(p) for p in preds)
    results = {s['token']: pred_to_global(p, s) for s, p in zip(VAL_SAMPLES, preds)}

    out_dir = f'/content/eval/{n_sweeps}sweep'
    os.makedirs(out_dir, exist_ok=True)
    result_path = f'{out_dir}/results_nusc.json'
    meta = dict(use_camera=False, use_lidar=True, use_radar=False, use_map=False, use_external=False)
    with open(result_path, 'w') as f:
        json.dump({'meta': meta, 'results': results}, f)

    ev = DetectionEval(nusc, config=EVAL_CFG, result_path=result_path, eval_set=EVAL_SET,
                       output_dir=out_dir, verbose=False)
    metrics, _ = ev.evaluate()
    print(f'{n_sweeps:2d} sweep: {len(VAL_SAMPLES)} keyframe, {n_checked} box du doan '
          f'(da kiem tra quy uoc box), {time.time() - t0:.0f}s')
    return ev, metrics.serialize()


EVALS, METRICS = {}, {}
for n in (0, N_SWEEPS):     # 0 = chi keyframe (cach cu), N_SWEEPS = giong luc huan luyen
    EVALS[n], METRICS[n] = evaluate(n)

# ----- Bang tong hop -----
cols = list(METRICS)
head = lambda n: f'{n + 1} lan quet'
gt_all = EVALS[cols[-1]].gt_boxes.all
N_GT = {c: sum(b.detection_name == c for b in gt_all) for c in CLASSES}
PRESENT = [c for c in CLASSES if N_GT[c] > 0]   # lop vang mat luon bi tinh AP = 0

print(f"\n{'chi so':28s}" + ''.join(f'{head(n):>14s}' for n in cols))
print('-' * (28 + 14 * len(cols)))
for key, label in [('mean_ap', 'mAP (cao = tot)'), ('nd_score', 'NDS (cao = tot)')]:
    print(f'{label:28s}' + ''.join(f'{METRICS[n][key]:14.3f}' for n in cols))
print(f'{f"mAP tren {len(PRESENT)} lop co mat":28s}' + ''.join(
    f"{np.mean([METRICS[n]['mean_dist_aps'][c] for c in PRESENT]):14.3f}" for n in cols))
TP_NAMES = {'trans_err': 'mATE vi tri (m)', 'scale_err': 'mASE kich thuoc (1-IoU)',
            'orient_err': 'mAOE goc xoay (rad)', 'vel_err': 'mAVE van toc (m/s)',
            'attr_err': 'mAAE thuoc tinh (1-acc)'}
for key, label in TP_NAMES.items():
    print(f'{label:28s}' + ''.join(f"{METRICS[n]['tp_errors'][key]:14.3f}" for n in cols))

print(f"\n{'AP theo lop':28s}" + ''.join(f'{head(n):>14s}' for n in cols) + f"{'so nhan':>9s}")
print('-' * (37 + 14 * len(cols)))
for c in CLASSES:
    print(f'{c:28s}' + ''.join(f"{METRICS[n]['mean_dist_aps'][c]:14.3f}" for n in cols) + f'{N_GT[c]:9d}')

## Cell 8 — Precision và recall theo ngưỡng điểm tin cậy

mAP ở trên không phụ thuộc ngưỡng. Nhưng khi **dùng model thật**, ta phải chọn một ngưỡng: box có điểm tin cậy dưới ngưỡng bị bỏ. Bảng này tính trên **cả 81 keyframe**, với cùng cách lọc nhãn gốc như bộ chấm điểm và cùng tiêu chí khớp 2 m.

- **P (precision)**: trong các box model đưa ra, bao nhiêu phần là đúng.
- **R (recall)**: trong các vật thật, model tìm ra bao nhiêu phần.

Ngưỡng thấp: tìm được nhiều hơn nhưng báo nhầm nhiều. Ngưỡng cao: ít báo nhầm nhưng bỏ sót nhiều.

In [ ]:
THRESHOLDS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]


def pr_by_threshold(ev, thresholds=THRESHOLDS, dist_th=2.0):
    """Precision/recall tren moi keyframe cua EVAL_SET, cung cach loc box cua bo cham diem.
    Khop tham lam di tu diem cao xuong, nen ket qua o nguong t chinh la phan dau cua lan khop
    voi moi box: chi can khop mot lan roi dem theo tung nguong."""
    tp, fp, n_gt = np.zeros(len(thresholds)), np.zeros(len(thresholds)), 0
    for token in ev.gt_boxes.sample_tokens:
        pred, gt = ev.pred_boxes[token], ev.gt_boxes[token]
        pred_tp, _, _ = match_boxes(pred, gt, dist_th)
        scores = np.array([b.detection_score for b in pred])
        for i, t in enumerate(thresholds):
            keep = scores >= t
            tp[i] += (pred_tp & keep).sum()
            fp[i] += (~pred_tp & keep).sum()
        n_gt += len(gt)
    return tp / np.maximum(tp + fp, 1), tp / max(n_gt, 1), n_gt


print(f"{'nguong':>7s}" + ''.join(f"{head(n) + ': P':>16s}{'R':>6s}" for n in cols))
print('-' * (7 + 22 * len(cols)))
PR = {n: pr_by_threshold(EVALS[n]) for n in cols}
for i, t in enumerate(THRESHOLDS):
    print(f'{t:7.2f}' + ''.join(f'{PR[n][0][i]:16.2f}{PR[n][1][i]:6.2f}' for n in cols))
print(f'\n(tren {len(VAL_SAMPLES)} keyframe, {PR[cols[-1]][2]} vat duoc cham; khop khi cung lop, tam cach < 2 m)')

## Cell 9 — Hàm vẽ hai hình cạnh nhau

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Colab web can renderer rieng cua plotly.
pio.renderers.default = 'colab'

EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]


def _box_lines(corners_list, color, label, texts=None):
    """Gom nhieu hop vao mot trace duy nhat cho nhe."""
    xs, ys, zs = [], [], []
    for k in corners_list:
        for i, j in EDGES:
            xs += [k[i, 0], k[j, 0], None]
            ys += [k[i, 1], k[j, 1], None]
            zs += [k[i, 2], k[j, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=3),
                        name=f'{label} ({len(corners_list)})',
                        hoverinfo='name' if texts is None else 'text',
                        text=None if texts is None else [t for t in texts for _ in range(36)])


def _points_trace(pts, max_points=60000):
    if len(pts) > max_points:
        pts = pts[np.random.default_rng(0).choice(len(pts), max_points, replace=False)]
    return go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
                        marker=dict(size=1, color=pts[:, 2], colorscale='Greys',
                                    opacity=0.55, showscale=False),
                        name='lidar', hoverinfo='skip', showlegend=False)


def show_pred_vs_gt(scene_name='scene-0103', index=20, score_thr=0.3, n_sweeps=N_SWEEPS,
                    dist_th=2.0, height=750, out_html=None):
    assert scene_name in VAL_SCENES, f'Chi xem duoc scene cua {EVAL_SET}: {VAL_SCENES}'
    sample, scene = get_sample(scene_name, index)
    ev = EVALS[n_sweeps]
    # Dung dung cac box bo cham diem da loc: trong pham vi cua lop, co diem lidar/radar,
    # bo xe dap/xe may dang dung trong bai do xe dap.
    gt = ev.gt_boxes[sample['token']]
    pred = [b for b in ev.pred_boxes[sample['token']] if b.detection_score >= score_thr]
    pred_tp, gt_hit, pairs = match_boxes(pred, gt, dist_th)

    n_tp, n_fp, n_fn = int(pred_tp.sum()), int((~pred_tp).sum()), int((~gt_hit).sum())
    precision = n_tp / max(len(pred), 1)
    recall = n_tp / max(len(gt), 1)

    pts = load_points(sample, n_sweeps)
    gt_k, pred_k = global_to_lidar_corners(gt, sample), global_to_lidar_corners(pred, sample)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                        subplot_titles=(f'NHAN GOC — {len(gt)} box',
                                        f'DU DOAN PointPillars — {len(pred)} box'))
    for col in (1, 2):
        fig.add_trace(_points_trace(pts), row=1, col=col)

    # Ben trai: nhan goc
    hit = [k for k, h in zip(gt_k, gt_hit) if h]
    miss = [k for k, h in zip(gt_k, gt_hit) if not h]
    if hit:
        fig.add_trace(_box_lines(hit, 'limegreen', 'GT duoc tim ra'), row=1, col=1)
    if miss:
        fig.add_trace(_box_lines(miss, 'orange', 'GT bi bo sot (FN)',
                                 texts=[b.detection_name for b, h in zip(gt, gt_hit) if not h]),
                      row=1, col=1)

    # Ben phai: du doan
    tp = [k for k, t in zip(pred_k, pred_tp) if t]
    fp = [k for k, t in zip(pred_k, pred_tp) if not t]
    if tp:
        fig.add_trace(_box_lines(tp, 'limegreen', 'Du doan dung (TP)'), row=1, col=2)
    if fp:
        fig.add_trace(_box_lines(fp, 'red', 'Bao nham (FP)',
                                 texts=[f'{b.detection_name} {b.detection_score:.2f}'
                                        for b, t in zip(pred, pred_tp) if not t]),
                      row=1, col=2)

    cam = dict(eye=dict(x=-1.3, y=-1.3, z=0.9))
    fig.update_layout(
        template='plotly_dark', height=height,
        title=(f"{scene['name']} keyframe {index} · {n_sweeps + 1} lan quet · nguong diem {score_thr} · "
               f"TP {n_tp} · FP {n_fp} · FN {n_fn} · "
               f"precision {precision:.2f} · recall {recall:.2f}"),
        scene=dict(aspectmode='data', camera=cam),
        scene2=dict(aspectmode='data', camera=cam),
        margin=dict(l=0, r=0, t=70, b=0))

    if out_html:
        fig.write_html(out_html)
        print('Da luu', out_html)
    fig.show()
    return dict(gt=gt, pred=pred, pred_tp=pred_tp, gt_hit=gt_hit, pairs=pairs,
                tp=n_tp, fp=n_fp, fn=n_fn, precision=precision, recall=recall)


print('Da dinh nghia xong ham ve.')

## Cell 10 — Xem một keyframe

Chỉ xem được hai scene của `mini_val`, vì chỉ những keyframe này mới được suy luận và chấm ở Cell 7. Box ở cả hai bên là **đúng các box mà bộ chấm điểm đã lọc**, nên vật ở quá xa hoặc không có điểm lidar sẽ không xuất hiện.

Đặt `N_SWEEPS_VIEW = 0` để xem kết quả của cách cũ (chỉ keyframe) trên cùng keyframe.

Hai khung 3D **xoay độc lập** với nhau. Bấm vào chú thích để ẩn/hiện từng nhóm box.

In [ ]:
SCENE         = 'scene-0916'  #@param ['scene-0103', 'scene-0916']
KEYFRAME      = 20            #@param {type:"slider", min:0, max:39, step:1}
SCORE_THR     = 0.3           #@param {type:"slider", min:0.05, max:0.9, step:0.05}
N_SWEEPS_VIEW = N_SWEEPS      # 0 = chi keyframe (cach cu)

res = show_pred_vs_gt(SCENE, KEYFRAME, score_thr=SCORE_THR, n_sweeps=N_SWEEPS_VIEW)

## Cell 11 — Bảng chi tiết theo từng lớp của keyframe đang xem

In [ ]:
import collections

rows = collections.defaultdict(lambda: [0, 0, 0])   # [TP, FP, FN]
for b, t in zip(res['pred'], res['pred_tp']):
    rows[b.detection_name][0 if t else 1] += 1
for b, h in zip(res['gt'], res['gt_hit']):
    if not h:
        rows[b.detection_name][2] += 1

print(f"{'lop':22s} {'TP':>4s} {'FP':>4s} {'FN':>4s}  {'precision':>9s} {'recall':>7s}")
print('-' * 60)
for n in sorted(rows, key=lambda k: -sum(rows[k])):
    tp, fp, fn = rows[n]
    p = tp / (tp + fp) if tp + fp else float('nan')
    r = tp / (tp + fn) if tp + fn else float('nan')
    print(f'{n:22s} {tp:4d} {fp:4d} {fn:4d}  {p:9.2f} {r:7.2f}')
print('-' * 60)
print(f"{'TONG':22s} {res['tp']:4d} {res['fp']:4d} {res['fn']:4d}  "
      f"{res['precision']:9.2f} {res['recall']:7.2f}")

print('\nCac cap khop duoc gan nhat (lop, diem tin cay, khoang cach tam):')
for pi, gi, d in sorted(res['pairs'], key=lambda x: x[2])[:10]:
    b = res['pred'][pi]
    print(f'  {b.detection_name:20s} diem {b.detection_score:.2f}  cach {d:.2f} m')

## Cell 12 (tuỳ chọn) — Lưu HTML và tải về

File HTML được lưu ở `/content/out`, rồi trình duyệt tự tải về máy. Nếu đặt `SAVE_TO_DRIVE = True`, file được chép thêm vào thư mục `pointpillars_out` trong My Drive.

Nếu trình duyệt chặn tải xuống, bấm biểu tượng thư mục ở thanh bên trái, vào `/content/out`, chuột phải file → **Download**.

In [ ]:
import os, shutil
from google.colab import files

SAVE_TO_DRIVE = False  #@param {type:"boolean"}
OUT_DIR = '/content/out'
os.makedirs(OUT_DIR, exist_ok=True)

out_html = (f'{OUT_DIR}/pred_vs_gt_{SCENE}_kf{KEYFRAME:02d}_'
            f'{N_SWEEPS_VIEW + 1}quet_thr{SCORE_THR:.2f}.html')
show_pred_vs_gt(SCENE, KEYFRAME, score_thr=SCORE_THR, n_sweeps=N_SWEEPS_VIEW, out_html=out_html)

if SAVE_TO_DRIVE:
    drive_dir = '/content/drive/MyDrive/pointpillars_out'
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(out_html, drive_dir)
    print('Da chep vao', drive_dir)
files.download(out_html)

---
## Cách đọc kết quả

| Màu | Ở hình bên trái (nhãn gốc) | Ở hình bên phải (dự đoán) |
|---|---|---|
| 🟢 Xanh lá | Vật thể model đã tìm ra | Dự đoán đúng (TP) |
| 🟠 Cam | Vật thể model **bỏ sót** (FN) | — |
| 🔴 Đỏ | — | **Báo nhầm** (FP) |

**Vì sao 11 lần quét tốt hơn nhiều:** model được huấn luyện với 11 lần quét chồng lên nhau. Chỉ đưa 1 lần quét, model nhận ít hơn khoảng 8 lần số điểm so với lúc học: vật ở xa gần như biến mất, và điểm tin cậy bị kéo xuống. Ngoài ra, không có nhiều lần quét thì model **không đoán được vận tốc**, nên mAVE, mAAE và NDS đều kém.

**So với con số công bố:** model zoo của MMDetection3D báo cáo mô hình này đạt khoảng **40 mAP, 53 NDS** trên **toàn bộ tập val** (150 scene, 6019 keyframe). `mini_val` chỉ có 2 scene, 81 keyframe và vắng 3 lớp, nên con số ở đây **không so trực tiếp được**. Dòng "mAP trên các lớp có mặt" gần với con số công bố hơn, nhưng vẫn dao động nhiều vì quá ít dữ liệu.

**Những giới hạn còn lại:**
- `barrier` và `traffic_cone` nhỏ, thường đứng sát nhau thành hàng, nên khó nhất.
- Muốn có số liệu đáng tin, cần chạy trên tập val đầy đủ của nuScenes (phải tải bản trainval).

## Xử lý sự cố

| Lỗi | Cách xử lý |
|---|---|
| Một bước ở Cell 2 báo `LOI` | Đọc log in ngay bên dưới. Log đầy đủ ở `/content/install.log`. Khắc phục rồi chạy lại Cell 2 |
| Cài hỏng giữa chừng, chạy lại vẫn lỗi | Chạy `!rm -rf /content/mmenv` rồi chạy lại Cell 2 |
| Không có GPU | Runtime → Change runtime type → T4 GPU → Save, rồi chạy lại từ Cell 1 |
| Cell 2 báo numpy đã đổi phiên bản | Runtime → Restart session, rồi chạy lại từ Cell 1 |
| `No module named 'nuscenes'` | Runtime đã bị thu hồi và cấp mới. Chạy lại từ Cell 1 |
| Cell 3 không tìm thấy `v1.0-mini.tgz` | Kiểm tra file đã upload xong lên Drive, sửa `TGZ_PATH` cho đúng rồi chạy lại Cell 3 |
| Cell 3 báo giải nén thất bại hoặc thiếu file | File trên Drive có thể hỏng: so dung lượng với bản gốc (khoảng 4.2 GB). Chạy `!rm -rf /content/nuscenes` rồi chạy lại Cell 3 |
| `Quy uoc box lech ... m so voi mmdet3d` | Phiên bản mmdet3d khác quy ước box. Dùng đúng mmdet3d 1.4.0 như Cell 2 |
| `CUDA out of memory` | Runtime → Restart session, rồi chạy lại từ Cell 3 |
| `KeyError: 'sweeps'` hoặc thiếu `dims_0` | File `/content/pp_infer.py` là bản cũ. Chạy lại Cell 5 |
| `mim download` lỗi mạng | Chạy lại cell, hoặc tải thủ công từ model zoo của MMDetection3D |
| `NameError` như `call_worker`, `nusc`, `EVALS` | Sau khi restart phải chạy lại lần lượt từ Cell 3 |
| Hình 3D không hiện | Chạy lại Cell 9 rồi Cell 10, hoặc mở file HTML ở Cell 12 bằng trình duyệt |
| Cài đặt hỏng hoàn toàn | Runtime → Disconnect and delete runtime, rồi chạy lại từ Cell 1 |

## Ghi chú

Trọng số được huấn luyện trên **700 scene train** của nuScenes đầy đủ. Cả hai scene của `mini_val` (0103, 0916) đều thuộc tập **val** của nuScenes, nên model **chưa từng thấy** chúng khi huấn luyện, và kết quả phản ánh đúng năng lực thật.